# Celestial Gardeners' Guild — Manual Trading 3

IMC Prosperity 4 manual trading challenge solver.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import collections
from itertools import product as iterproduct

## Problem Statement

**Setup:**
- Trade against counterparties with reserve prices uniformly distributed on the discrete grid $\{670, 675, 680, \ldots, 915, 920\}$ — 51 values, increments of 5.
- Resell all acquired product tomorrow at fair price **920**.
- Submit two bids: $b_1$ (low) and $b_2$ (high), with $b_1 < b_2$, both on the grid.

**Trade rules for each counterparty with reserve price $r$:**
1. If $b_1 \geq r$: trade at $b_1$, profit $= 920 - b_1$
2. If $b_1 < r \leq b_2$ and $b_2 > \bar{b}_2$: trade at $b_2$, profit $= 920 - b_2$ (clean trade)
3. If $b_1 < r \leq b_2$ and $b_2 \leq \bar{b}_2$: trade at $b_2$, profit $= (920 - b_2) \times \left(\frac{920 - \bar{b}_2}{920 - b_2}\right)^3$ (penalized)
4. If $r > b_2$: no trade

where $\bar{b}_2$ is the mean of all other players' second bids (unknown — must be modeled).

The penalty in rule 3 simplifies to $\frac{(920 - \bar{b}_2)^3}{(920 - b_2)^2}$.

## Grid & Distribution

In [ ]:
# Bid grid
GRID = np.arange(670, 921, 5)  # 670, 675, ..., 920
FAIR = 920
N_GRID = len(GRID)
print(f"Grid: {GRID[0]} to {GRID[-1]}, {N_GRID} values, step 5")

# Reserve prices are uniform on GRID => each has probability 1/51
P_EACH = 1.0 / N_GRID

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(GRID, [P_EACH] * N_GRID, width=4, color='steelblue', alpha=0.7)
ax.set_xlabel('Reserve price')
ax.set_ylabel('Probability')
ax.set_title('Uniform distribution of reserve prices')
plt.tight_layout()
plt.show()

## Part A: Solve $b_1$ in Isolation

Ignoring $b_2$ entirely, the expected profit from a single bid $b_1$ is:
$$E[\pi(b_1)] = \Pr(r \leq b_1) \times (920 - b_1) = \frac{\#\{r \in \text{grid} : r \leq b_1\}}{51} \times (920 - b_1)$$

In [ ]:
def prob_leq(b):
    """Probability that reserve price r <= b, on the discrete grid."""
    count = np.sum(GRID <= b)
    return count / N_GRID

def prob_between(lo, hi):
    """Probability that lo < r <= hi, on the discrete grid."""
    count = np.sum((GRID > lo) & (GRID <= hi))
    return count / N_GRID

# Grid search for b1 alone
profits_b1 = np.array([(FAIR - b) * prob_leq(b) for b in GRID])
best_idx = np.argmax(profits_b1)
best_b1_alone = GRID[best_idx]

print(f"Optimal b1 (isolation): {best_b1_alone}")
print(f"Expected profit per counterparty: {profits_b1[best_idx]:.2f}")
print(f"Prob(trade): {prob_leq(best_b1_alone):.3f}, margin: {FAIR - best_b1_alone}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(GRID, profits_b1, 'o-', markersize=4, color='steelblue')
ax.axvline(best_b1_alone, color='red', linestyle='--', label=f'Optimal b1 = {best_b1_alone}')
ax.set_xlabel('b1')
ax.set_ylabel('Expected profit per counterparty')
ax.set_title('Part A: Single-bid expected profit')
ax.legend()
plt.tight_layout()
plt.show()

## Part B: Joint $(b_1, b_2)$ Optimization — No Penalty

Ignoring the $b_2$ penalty (assuming we always beat the field), expected profit is:
$$E[\pi] = \Pr(r \leq b_1)(920 - b_1) + \Pr(b_1 < r \leq b_2)(920 - b_2)$$

In [ ]:
def expected_profit_no_penalty(b1, b2):
    """Expected profit per counterparty, ignoring the b2 penalty."""
    return (FAIR - b1) * prob_leq(b1) + (FAIR - b2) * prob_between(b1, b2)

# Vectorized grid search
profit_matrix = np.full((N_GRID, N_GRID), np.nan)
best_val = -np.inf
best_pair = (GRID[0], GRID[0])

for i, b1 in enumerate(GRID):
    for j, b2 in enumerate(GRID):
        if b2 <= b1:
            continue
        val = expected_profit_no_penalty(b1, b2)
        profit_matrix[i, j] = val
        if val > best_val:
            best_val = val
            best_pair = (b1, b2)

print(f"Optimal (b1, b2) [no penalty]: {best_pair}")
print(f"Expected profit per counterparty: {best_val:.2f}")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(profit_matrix, origin='lower', aspect='auto',
               extent=[GRID[0], GRID[-1], GRID[0], GRID[-1]],
               cmap='viridis')
ax.set_xlabel('b2 (high bid)')
ax.set_ylabel('b1 (low bid)')
ax.set_title('Part B: Expected profit heatmap (no penalty)')
ax.plot(best_pair[1], best_pair[0], 'r*', markersize=15, label=f'Optimum {best_pair}')
ax.legend()
plt.colorbar(im, label='Expected profit per counterparty')
plt.tight_layout()
plt.show()

## Part C: Level-k Reasoning for $b_2$

The penalty depends on $\bar{b}_2$ (the field's average second bid). We use iterated best-response:

- **Level-0**: Ignore penalty entirely → use Part B optimum as $b_2$.
- **Level-1**: Assume everyone plays level-0, so $\bar{b}_2 = b_2^{(0)}$. Best-respond.
- **Level-k**: Best-respond to level-(k-1).

When $b_2 \leq \bar{b}_2$, the per-trade profit becomes $(920 - \bar{b}_2)^3 / (920 - b_2)^2$ instead of $(920 - b_2)$.
When $b_2 > \bar{b}_2$, no penalty (clean trade, profit = $920 - b_2$).

In [ ]:
def expected_profit_with_penalty(b1, b2, avg_b2):
    """Expected profit per counterparty, accounting for the b2 penalty.
    
    Parameters
    ----------
    b1 : int
        Low bid.
    b2 : int
        High bid.
    avg_b2 : float
        Assumed average second bid of the field.
    
    Returns
    -------
    float
        Expected profit per counterparty.
    """
    profit_b1 = (FAIR - b1) * prob_leq(b1)
    p_b2_range = prob_between(b1, b2)
    
    if b2 > avg_b2:
        # Clean trade — no penalty
        profit_b2 = (FAIR - b2) * p_b2_range
    else:
        # Penalized: multiplier = ((920 - avg_b2) / (920 - b2))^3
        if FAIR - b2 == 0:
            profit_b2 = 0.0
        else:
            penalty_multiplier = ((FAIR - avg_b2) / (FAIR - b2)) ** 3
            profit_b2 = (FAIR - b2) * penalty_multiplier * p_b2_range
    
    return profit_b1 + profit_b2


def best_respond(avg_b2):
    """Find the (b1, b2) pair that maximizes expected profit given assumed avg_b2.
    
    Parameters
    ----------
    avg_b2 : float
        Assumed average second bid of the field.
    
    Returns
    -------
    tuple
        (best_b1, best_b2, best_profit)
    """
    best_val = -np.inf
    best = (GRID[0], GRID[-1])
    for b1 in GRID:
        for b2 in GRID:
            if b2 <= b1:
                continue
            val = expected_profit_with_penalty(b1, b2, avg_b2)
            if val > best_val:
                best_val = val
                best = (b1, b2)
    return best[0], best[1], best_val


# Level-k tower
MAX_LEVELS = 10
levels = []

# Level 0: no penalty consideration
level0_b1, level0_b2 = best_pair
level0_profit = best_val
levels.append({'level': 0, 'b1': level0_b1, 'b2': level0_b2, 
               'profit': round(level0_profit, 4), 'avg_b2_assumed': 'N/A'})

prev_b2 = float(level0_b2)
for k in range(1, MAX_LEVELS + 1):
    b1_k, b2_k, profit_k = best_respond(prev_b2)
    levels.append({'level': k, 'b1': b1_k, 'b2': b2_k,
                   'profit': round(profit_k, 4), 'avg_b2_assumed': prev_b2})
    
    # Check convergence
    if b2_k == prev_b2:
        print(f"Converged at level {k}")
        break
    
    # Check cycle
    past_b2s = [l['b2'] for l in levels[:-1]]
    if b2_k in past_b2s:
        print(f"Cycle detected at level {k}: b2={b2_k} appeared before")
        break
    
    prev_b2 = float(b2_k)

# Print table
print(f"\n{'Level':<7} {'b1':<6} {'b2':<6} {'Assumed avg_b2':<16} {'E[profit]':<10}")
print('-' * 50)
for l in levels:
    avg_str = f"{l['avg_b2_assumed']:.0f}" if isinstance(l['avg_b2_assumed'], float) else l['avg_b2_assumed']
    print(f"{l['level']:<7} {l['b1']:<6} {l['b2']:<6} {avg_str:<16} {l['profit']:<10.4f}")

## Part D: Monte Carlo Robustness Check

Simulate 1000 realizations with $n = 5000$ counterparties. Model the field as a mixture of level-k players to estimate $\bar{b}_2$.

In [ ]:
# ====== MIXTURE WEIGHTS — EASY TO TWEAK ======
# Maps level -> weight. Weights are normalized automatically.
FIELD_MIXTURE = {
    0: 0.30,   # 30% play level-0 (naive, ignore penalty)
    1: 0.40,   # 40% play level-1
    2: 0.20,   # 20% play level-2
    'naive': 0.10,  # 10% bid naively at 920 (maximum)
}
# =============================================

def field_avg_b2(mixture, levels_table):
    """Compute the field's average b2 given a mixture of level-k players.
    
    Parameters
    ----------
    mixture : dict
        Maps level (int or 'naive') to weight.
    levels_table : list of dict
        Level-k tower results.
    
    Returns
    -------
    float
        Weighted average b2 of the field.
    """
    level_b2 = {l['level']: l['b2'] for l in levels_table}
    total_w = sum(mixture.values())
    avg = 0.0
    for k, w in mixture.items():
        if k == 'naive':
            avg += (w / total_w) * 920
        elif k in level_b2:
            avg += (w / total_w) * level_b2[k]
        else:
            # Use highest available level
            max_level = max(l['level'] for l in levels_table)
            avg += (w / total_w) * level_b2[max_level]
    return avg


def simulate_profit(b1, b2, avg_b2, n_counterparties, n_sims):
    """Monte Carlo simulation of total profit.
    
    Parameters
    ----------
    b1 : int
        Low bid.
    b2 : int
        High bid.
    avg_b2 : float
        Assumed field average b2.
    n_counterparties : int
        Number of counterparties per simulation.
    n_sims : int
        Number of simulations.
    
    Returns
    -------
    ndarray
        Array of total profits, shape (n_sims,).
    """
    # Sample reserve prices: uniform on GRID
    reserves = np.random.choice(GRID, size=(n_sims, n_counterparties))
    
    # Trades at b1
    mask_b1 = reserves <= b1
    profit_b1 = mask_b1.astype(float) * (FAIR - b1)
    
    # Trades at b2
    mask_b2 = (reserves > b1) & (reserves <= b2)
    if b2 > avg_b2:
        per_trade_b2 = FAIR - b2
    else:
        if FAIR - b2 == 0:
            per_trade_b2 = 0.0
        else:
            per_trade_b2 = (FAIR - b2) * ((FAIR - avg_b2) / (FAIR - b2)) ** 3
    profit_b2 = mask_b2.astype(float) * per_trade_b2
    
    return (profit_b1 + profit_b2).sum(axis=1)


# Compute field avg_b2
avg_b2_field = field_avg_b2(FIELD_MIXTURE, levels)
print(f"Field average b2 (mixture model): {avg_b2_field:.1f}")

# Candidates from the level-k tower
candidates = list({(l['b1'], l['b2']) for l in levels})
candidates.sort()

N_COUNTERPARTIES = 5000
N_SIMS = 1000
np.random.seed(42)

fig, axes = plt.subplots(len(candidates), 1, figsize=(12, 3 * len(candidates)), sharex=True)
if len(candidates) == 1:
    axes = [axes]

print(f"\n{'(b1, b2)':<16} {'Mean PnL':>10} {'Std':>10} {'5th %ile':>10} {'95th %ile':>10}")
print('-' * 60)

for ax, (b1, b2) in zip(axes, candidates):
    pnls = simulate_profit(b1, b2, avg_b2_field, N_COUNTERPARTIES, N_SIMS)
    mean_pnl = pnls.mean()
    std_pnl = pnls.std()
    p5, p95 = np.percentile(pnls, [5, 95])
    
    print(f"({b1}, {b2}){'':<8} {mean_pnl:>10.0f} {std_pnl:>10.0f} {p5:>10.0f} {p95:>10.0f}")
    
    ax.hist(pnls, bins=40, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(mean_pnl, color='red', linestyle='--', label=f'Mean = {mean_pnl:.0f}')
    ax.set_title(f'(b1={b1}, b2={b2})')
    ax.legend()

axes[-1].set_xlabel('Total PnL')
fig.suptitle(f'Monte Carlo PnL distributions (n={N_COUNTERPARTIES}, {N_SIMS} sims)', y=1.01)
plt.tight_layout()
plt.show()

## Part E: Sensitivity Analysis

Vary the assumed field mixture and see how the optimal $(b_1, b_2)$ shifts.

In [ ]:
def optimal_for_mixture(mixture, levels_table):
    """Find optimal (b1, b2) given a field mixture.
    
    Parameters
    ----------
    mixture : dict
        Maps level (int or 'naive') to weight.
    levels_table : list of dict
        Level-k tower results.
    
    Returns
    -------
    tuple
        (best_b1, best_b2, best_profit, avg_b2)
    """
    avg = field_avg_b2(mixture, levels_table)
    best_val = -np.inf
    best = (GRID[0], GRID[-1])
    for b1 in GRID:
        for b2 in GRID:
            if b2 <= b1:
                continue
            val = expected_profit_with_penalty(b1, b2, avg)
            if val > best_val:
                best_val = val
                best = (b1, b2)
    return best[0], best[1], best_val, avg


# Define mixture scenarios
scenarios = [
    ('Mostly naive',      {0: 0.5, 1: 0.2, 2: 0.1, 'naive': 0.2}),
    ('Balanced',          {0: 0.3, 1: 0.4, 2: 0.2, 'naive': 0.1}),
    ('Sophisticated',     {0: 0.1, 1: 0.3, 2: 0.4, 'naive': 0.2}),
    ('Very sophisticated',{0: 0.05, 1: 0.15, 2: 0.5, 'naive': 0.3}),
    ('All naive',         {0: 0.0, 1: 0.0, 2: 0.0, 'naive': 1.0}),
    ('All level-0',       {0: 1.0, 1: 0.0, 2: 0.0, 'naive': 0.0}),
]

print(f"{'Scenario':<22} {'b1':<6} {'b2':<6} {'Field avg_b2':<14} {'E[profit]':<10}")
print('-' * 60)
results_sensitivity = []
for name, mix in scenarios:
    b1, b2, profit, avg = optimal_for_mixture(mix, levels)
    print(f"{name:<22} {b1:<6} {b2:<6} {avg:<14.1f} {profit:<10.4f}")
    results_sensitivity.append((name, b1, b2, profit, avg))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = [r[0] for r in results_sensitivity]
b1s = [r[1] for r in results_sensitivity]
b2s = [r[2] for r in results_sensitivity]
profits = [r[3] for r in results_sensitivity]

x = np.arange(len(names))
ax1.bar(x - 0.15, b1s, 0.3, label='b1', color='steelblue')
ax1.bar(x + 0.15, b2s, 0.3, label='b2', color='coral')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=30, ha='right')
ax1.set_ylabel('Bid value')
ax1.set_title('Optimal bids by scenario')
ax1.legend()

ax2.bar(x, profits, color='seagreen')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=30, ha='right')
ax2.set_ylabel('Expected profit per counterparty')
ax2.set_title('Expected profit by scenario')

plt.tight_layout()
plt.show()

## Final Recommendation

**Summary of findings:**

1. **Part A** (single bid): The unconstrained optimum for $b_1$ alone is around 795, capturing ~50% of counterparties with margin 125.

2. **Part B** (two bids, no penalty): Adding a second bid improves expected profit substantially. The optimum shifts $b_1$ down and adds $b_2$ to capture higher-reserve counterparties.

3. **Part C** (level-k reasoning): The penalty for being below the field's $\bar{b}_2$ creates a beauty-contest dynamic. The level-k tower typically converges or cycles within a few iterations.

4. **Part D** (Monte Carlo): With 5000 counterparties the realized profit is tightly concentrated around the expectation — the law of large numbers works in our favor.

5. **Part E** (sensitivity): The optimal $b_1$ is robust across field assumptions. The optimal $b_2$ shifts depending on where we think the field plays, but the profit differences between nearby $b_2$ values are small.

**Recommended strategy:** Use the level-1 or level-2 result from Part C. The exact bids depend on the level-k tower output above. When in doubt, err toward a slightly higher $b_2$ — the penalty for being below the mean is cubic, while the cost of slightly overpaying is linear.